# L06 · DQN과 function approximation

## Goal

- replay와 target network의 역할을 설명한다
- target gradient를 끊는다
- TD loss의 shape을 확인한다

## Setup

이 cell은 CPU·seed·offline 상태와 split hash를 먼저 고정합니다. toy 연산은 결정론적인 CPU 연산만 쓰며, package trainer의 전역 결정론 기본값은 유지합니다.

In [1]:
import hashlib, json, os, platform, random, sys
from pathlib import Path
os.environ.setdefault("TORCH_DEVICE_BACKEND_AUTOLOAD", "0")
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run this notebook inside the RL-study repository")
sys.path.insert(0, str(ROOT / "src"))
import torch
from rl_study import __version__
from rl_study.data import build_tiny_reasoning
from rl_study.runtime import resolve_device, seed_everything
# These notebooks use only deterministic CPU toy kernels.  PyTorch 2.13's global
# guard imports the full Inductor stack, so keep the package's strict default for
# trainers while avoiding that unrelated startup cost in fresh teaching kernels.
seed_everything(42, deterministic=False)
random.seed(42)
language = os.environ.get("RL_STUDY_NOTEBOOK_LANGUAGE", "ko")
resolution = resolve_device("cpu")
dataset = build_tiny_reasoning(seed=42)
config_hash = "sha256:" + hashlib.sha256(b"L06:toy:42").hexdigest()
print(f"lesson=L06 language={language} profile=toy")
print("seed=42 network_required=False deterministic_scope=seeded_cpu_toy")
print(f"python={platform.python_version()} rl_study={__version__} torch={torch.__version__}")
print(f"requested_device=cpu resolved_device={resolution.resolved} fallback_used={resolution.fallback_used}")
print(f"config_hash={config_hash} data_split_hash={dataset.split_hash}")

lesson=L06 language=ko profile=toy
seed=42 network_required=False deterministic_scope=seeded_cpu_toy
python=3.10.12 rl_study=0.1.0.dev0 torch=2.13.0
requested_device=cpu resolved_device=cpu fallback_used=False
config_hash=sha256:afdf5bdf3ec7ea09c983beb03612c9488e8db611ddbdd80e0b7b079f922f3d29 data_split_hash=sha256:f238657bbf6c0a112debf7ef3ffafb452c14308dfb5ce57d9abe4f77ac1deedd


## Steps

### 1. 현재 위치와 핵심 식

⏱ 5분 · 1/3 section · [필수/CORE]

현재 위치: MC·TD·Q-learning → **DQN** → policy gradient

$$L(\theta)=\mathbb{E}\left[\left(Q_\theta(s,a)-\operatorname{stopgrad}(r+\gamma\max_{a'}Q_{\bar\theta}(s',a'))\right)^2\right]$$

DQN은 Q table을 network로 근사합니다. replay buffer는 연속 sample의 상관을 줄이고 재사용하며, target network는 target이 매 gradient step마다 함께 움직이지 않게 늦게 동기화합니다.

### 2. 작은 숫자로 실행

⏱ 6분 · 2/3 section · [필수/CORE]

**먼저 예측:** loss.backward 뒤 target network parameter에 gradient가 생겨야 할까요? 20초 동안 답을 적은 뒤 실행하세요.

<details><summary>정답 보기</summary>아니요. target은 학습 신호의 숫자만 제공하며 optimizer 소유가 아닙니다.</details>

In [2]:
from rl_study.algorithms.dqn import DQNBatch, DQNNetwork, dqn_loss, hard_update
policy_net = DQNNetwork(16, 4)
target_net = DQNNetwork(16, 4)
hard_update(target_net, policy_net)
dqn_batch = DQNBatch(
    states=torch.tensor([0, 1]), actions=torch.tensor([1, 2]),
    rewards=torch.tensor([-0.01, 1.0]), next_states=torch.tensor([1, 15]),
    terminated=torch.tensor([False, True]), truncated=torch.tensor([False, False])
)
dqn_output = dqn_loss(policy_net, target_net, dqn_batch, gamma=0.99)
dqn_output.loss.backward()
target_has_gradient = any(p.grad is not None for p in target_net.parameters())
print({"loss": round(float(dqn_output.loss.detach()), 4),
       "targets": dqn_output.targets.tolist(),
       "target_has_gradient": target_has_gradient})

{'loss': 0.3481, 'targets': [0.11680736392736435, 1.0], 'target_has_gradient': False}


### 3. 구현 해부

⏱ 6분 · 3/3 section · [심화/DEEP DIVE]

**왜 이렇게 구현했나:** 여기서는 작은 batch로 target detach 계약을 먼저 검증합니다. Double DQN은 action 선택과 평가를 나눠 과대추정을 줄이는 대안이지만 기본 gradient ownership은 같습니다.

**흔한 함정:** target tensor만 detach해도 target network를 optimizer에 넣으면 이후 다른 loss에서 움직일 수 있습니다. frozen parameter와 optimizer membership도 함께 검사해야 합니다. 회귀 test: `test_dqn_target_detached`.

**쉬어가기:** 지금 출력한 한 값만 설명할 수 있으면 다음 cell로 가세요.

## Checks

In [3]:
assert not target_has_gradient and torch.isfinite(dqn_output.loss)
print("checks=passed")

checks=passed


**회상 문제:** replay와 target network가 각각 줄이려는 불안정성은 어떻게 다른가요? 1~2문장으로 답하세요.

## 내가 자주 틀리는 것

- loss가 유한하면 구현도 맞다고 생각한다.
- `terminated`와 `truncated`, prompt와 action을 합친다.
- 한 seed의 작은 결과를 알고리즘 순위로 확대한다.

## 60초 요약

- **실행 결론:** loss와 target은 유한했고 `target_has_gradient=False`였습니다. 이 출력은 update 방향보다 frozen target 경계를 검증합니다.
- 실제 확인: `test_dqn_target_detached`.
- 출력은 고정 seed의 toy 실행이며 논문 규모 결과가 아닙니다.

## Next Steps

1. L07에서는 action value를 맞추는 대신 policy 확률을 reward 방향으로 직접 움직입니다.
2. `[필수/CORE]` assertion을 한 번 깨뜨리고 오류를 읽습니다.
3. package test를 열어 notebook의 작은 식과 production guard를 연결합니다.

## Sources

- `dqn-2013` — `docs/sources.yml`